# CANguard -- Feature Engineering (thin orchestrator)

Drives `canguard.features`. The window/feature logic lives in the library
and is covered by `tests/test_features*.py` and `tests/test_per_id.py`.


## 1. Setup


In [ ]:
from pathlib import Path

from canguard.data import get_loader
from canguard.features import (
    BEHAVIORAL_FEATURES_V1 as FEATURES,
)
from canguard.features import (
    FeaturePipeline,
    fit_known_ids_on_normal_prefix,
)
from canguard.features.groups import GROUP_DLC, GROUP_FLAT_BYTE, GROUP_IAT, GROUP_OTHER

DATA_DIR = Path('HCRL Car-Hacking')
SAMPLE_SIZE = 60000
WINDOW_SIZE = 30


## 2. Build per-ID feature tables (all four datasets)
Window label_policy is 'any'; presence features are OFF for the main path.


In [ ]:
samples = {
    name: get_loader('hcrl', DATA_DIR / f'{name}_dataset.csv').load(sample_size=SAMPLE_SIZE)
    for name in ['DoS', 'Fuzzy', 'RPM', 'gear']
}
feature_tables = {}
for name, df in samples.items():
    known = fit_known_ids_on_normal_prefix(df)
    pipe = FeaturePipeline(window_size=WINDOW_SIZE, known_ids=known)
    ft = pipe.process_dataframe(df)
    pipe.reset()
    feature_tables[name] = ft
    n_att = int(ft['is_attack'].sum())
    print(f'{name:>6s}: {len(ft):>6,} windows  ({n_att} attack, {n_att/len(ft)*100:.1f}%)')


## 3. Feature groups (canonical, from the library)


In [ ]:
print('V1 behavioral features:', len(FEATURES))
print('  IAT:', GROUP_IAT)
print('  Flat byte:', GROUP_FLAT_BYTE)
print('  DLC:', GROUP_DLC)
print('  Other:', GROUP_OTHER)
print('\nFull table columns:', list(feature_tables['RPM'].columns))
